In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# -----------------------------
# Prepare your dataset
# -----------------------------
heights_cm = [165.1, 167.6, 170.2, 172.7, 175.3, 177.8, 180.3, 182.9, 185.4, 188.0, 190.5, 193.0, 195.6]

weight_lbs_ranges = [
    (130, 150), (150, 160), (160, 170), (170, 180), (180, 190),
    (190, 200), (200, 210), (210, 220), (220, 240), (240, 270),
    (270, 300), (300, 340)
]
weights_kg = [((low + high) / 2) * 0.453592 for low, high in weight_lbs_ranges]

sizes = [
    ['XS', 'S', 'M', None, None, None, None, None, None, None, None, None],
    ['XS', 'S', 'M', 'L', None, None, None, None, None, None, None, None],
    ['XS', 'S', 'M', 'L', 'L', None, None, None, None, None, None, None],
    ['XS', 'S', 'M', 'L', 'L', 'XL', None, None, None, None, None, None],
    ['XS', 'S', 'M', 'M', 'L', 'XL', '2XL', None, None, None, None, None],
    [None, 'S', 'M', 'M', 'L', 'L', '2XL', '2XL', None, None, None, None],
    [None, None, 'M', 'M', 'L', 'L', 'XL', '2XL', '3XL', None, None, None],
    [None, None, None, 'M', 'L', 'L', 'XL', '2XL', '3XL', '4XL', None, None],
    [None, None, None, None, 'L', 'L', 'XL', '2XL', '3XL', '4XL', '5XL', None],
    [None, None, None, None, None, 'L', 'XL', '2XL', '3XL', '4XL', '5XL', '6XL'],
    [None, None, None, None, None, None, 'XL', '2XL', '2XL', '3XL', '5XL', '6XL'],
    [None, None, None, None, None, None, None, '2XL', '2XL', '3XL', '4XL', '5XL'],
    [None, None, None, None, None, None, None, None, '2XL', '3XL', '4XL', '5XL'],
]


In [3]:
# Build dataset
data_rows = []
for i, height in enumerate(heights_cm):
    for j, weight in enumerate(weights_kg):
        size = sizes[i][j]
        if size is not None:
            data_rows.append([height, weight, size])

df = pd.DataFrame(data_rows, columns=['Height_cm', 'Weight_kg', 'Size'])

In [4]:
# Encode target
size_encoder = LabelEncoder()
df['Size_encoded'] = size_encoder.fit_transform(df['Size'])

# Features and target
X = df[['Height_cm', 'Weight_kg']]
y = df['Size_encoded']

In [10]:
# Standardize features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [45]:
# -----------------------------
# Multinomial Logistic Regression
# -----------------------------
import warnings
warnings.filterwarnings("ignore")

# logreg = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=100) # Standard Log Reg ML model, Accuracy 34.6%
logreg = LogisticRegression(multi_class='multinomial', solver='saga', penalty=None, max_iter=1000) # Use of penalty to prevent overfitting and avoids regularization, Accuracy 87.1%

logreg.fit(X_scaled, y)

# Evaluate with cross-validation
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(logreg, X_scaled, y, cv=10, scoring='accuracy')
model_accuracy = cv_scores.mean()
print(f"Cross-validated accuracy: {cv_scores.mean():.3f}")


Cross-validated accuracy: 0.871


In [46]:
# -----------------------------
# Prediction function
# -----------------------------
def predict_tshirt_size(height_cm, weight_kg):
    input_scaled = scaler.transform([[height_cm, weight_kg]])
    pred_encoded = logreg.predict(input_scaled)[0]
    return size_encoder.inverse_transform([pred_encoded])[0]



In [ ]:
# # -----------------------------
# # User input for prediction
# # -----------------------------
# def main():
#     try:
#         height = float(input("Enter your height in centimeters (e.g., 175): ").strip())
#         weight = float(input("Enter your weight in kilograms (e.g., 70): ").strip())
#     except ValueError:
#         print("Invalid input. Please enter numeric values for height and weight.")
#         return

#     pred_size = predict_tshirt_size(height, weight)
#     print(f"Predicted T-shirt size for {height} cm and {weight} kg: {pred_size}")

# if __name__ == "__main__":
#     main()


In [ ]:
# -----------------------------
# User input for prediction
# -----------------------------
def main():
    try:
        height = float(input("Enter your height in centimeters (e.g., 175): ").strip())
        weight = float(input("Enter your weight in kilograms (e.g., 70): ").strip())
    except ValueError:
        print("Invalid input. Please enter numeric values for height and weight.")
        return

    pred_size = predict_tshirt_size(height, weight)
    print(f"Predicted T-shirt size for {height} cm and {weight} kg: {pred_size}")
    print(f"Estimated model prediction accuracy: {model_accuracy*100:.2f}%")
    
if __name__ == "__main__":
    main()


Predicted T-shirt size for 170.0 cm and 70.0 kg: S
Estimated model prediction accuracy: 87.14%
